# vitalAperiodic

The vitalAperiodic table provides invasive vital sign data which is interfaced into eCareManager at irregular intervals. Unlike most tables in eICU-CRD, vitalAperiodic does not use an entity-attribute-value model, but rather has an individual column to capture each data element. Columns available include:

* Blood pressures: nonInvasiveSystolic, nonInvasiveDiastolic, nonInvasiveMean
* Cardiac output measures: cardiacOutput, cardiacInput
* Systemic circulation measures: svr, svri, pvr, pvri
* Pulmonary pressures: pulmonary artery occlusion pressure (paop)

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psycopg2
import getpass
import pdvega

# for configuring connection 
from configobj import ConfigObj
import os
from loguru import logger

import holoviews as hv
from holoviews import opts
hv.extension("bokeh")


%matplotlib inline

In [ ]:
# Create a database connection using settings from config file
config = '../db/config.ini'

# connection info
conn_info = dict()
if os.path.isfile(config):
    config = ConfigObj(config)
    conn_info["sqluser"] = config['username']
    conn_info["sqlpass"] = config['password']
    conn_info["sqlhost"] = config['host']
    conn_info["sqlport"] = config['port']
    conn_info["dbname"] = config['dbname']
    conn_info["schema_name"] = config['schema_name']
else:
    conn_info["sqluser"] = 'postgres'
    conn_info["sqlpass"] = ''
    conn_info["sqlhost"] = 'localhost'
    conn_info["sqlport"] = 5432
    conn_info["dbname"] = 'eicu'
    conn_info["schema_name"] = 'public,eicu_crd'

# Connect to the eICU database
print('Database: {}'.format(conn_info['dbname']))
print('Username: {}'.format(conn_info["sqluser"]))
if conn_info["sqlpass"] == '':
    # try connecting without password, i.e. peer or OS authentication
    try:
        if (conn_info["sqlhost"] == 'localhost') & (conn_info["sqlport"] == '5432'):
            con = psycopg2.connect(dbname=conn_info["dbname"],
                                   user=conn_info["sqluser"])
        else:
            con = psycopg2.connect(dbname=conn_info["dbname"],
                                   host=conn_info["sqlhost"],
                                   port=conn_info["sqlport"],
                                   user=conn_info["sqluser"])
    except:
        conn_info["sqlpass"] = getpass.getpass('Password: ')

        con = psycopg2.connect(dbname=conn_info["dbname"],
                               host=conn_info["sqlhost"],
                               port=conn_info["sqlport"],
                               user=conn_info["sqluser"],
                               password=conn_info["sqlpass"])
query_schema = 'set search_path to ' + conn_info['schema_name'] + ';'

In [ ]:
from sqlalchemy import create_engine

db_credentials_url = "".join(
    [config['username'], ":", config['password'], "@", config['host'], ":", config['port'], "/", config['dbname']])
database_url = "".join(["postgresql+psycopg2://", db_credentials_url])

logger.info(f"Connecting to the database with database_url={database_url}")

engine = create_engine(database_url)

## Examine a single patient

In [ ]:
patientunitstayid = 145467

In [ ]:
query = query_schema + """
select *
from vitalaperiodic
where patientunitstayid = {}
order by observationoffset
""".format(patientunitstayid)

df = pd.read_sql_query(query, engine)
df.set_index('observationoffset', inplace=True)
df.sort_index(inplace=True)
df = df.reset_index()

df.head()

In [ ]:
# list of columns to plot
vitals = ['noninvasivesystolic', 'noninvasivediastolic', 'noninvasivemean',
          #'paop',
          'cardiacoutput', 'cardiacinput',
          'pvr', 'pvri']
# we exclude  'svr', 'svri' from the plot as their scale is too high
# we exclude 'paop' as it's all none

In [ ]:

x = df.index
y = df["cardiacoutput"]
y_mult = df[vitals]
#df.plot(x=df.index, y="noninvasivesystolic")
cardiac_curve = hv.Curve((x, y))
cardiac_curve

In [ ]:
cardiac_curve_2 = hv.Curve(df, ("observationoffset", "time"), ("cardiacoutput", "cardiacoutput"))
cardiac_curve_2

In [ ]:
holo_ds = hv.Dataset(df).sort("observationoffset")
vitals_curve = holo_ds.to(hv.Curve, "observationoffset", vitals)
display(vitals_curve)

In [ ]:
columns = ["observationoffset", 'noninvasivesystolic', 'noninvasivediastolic', 'noninvasivemean', 'cardiacoutput',
           'cardiacinput',
           'pvr', 'pvri']

long_df = df[columns].melt(id_vars='observationoffset',
                           var_name='Variable',
                           value_name='Value')

plot = hv.Dataset(long_df).to(hv.Curve, 'observationoffset', 'Value', groupby='Variable').overlay()

xmin = df["observationoffset"].min()
xmax = df["observationoffset"].max()
plot.opts(opts.Curve(width=1080, height=720, tools=['hover'], autorange='x', xlim=(xmin, xmax)))


In [ ]:
df.head()

In [ ]:
df.set_index('observationoffset', inplace=True)
out_of_range_columns = ["vitalaperiodicid", "patientunitstayid", "svri", "svr", "paop"]
df = df.drop(columns=out_of_range_columns, axis=1)

overlay_regular = hv.Overlay(
    [hv.Curve((df.index, df[col]), label=str(col)) * hv.Scatter((df.index, df[col])).opts(size=5) for col in
     df.columns])
overlay_regular.opts(
    'Curve', tools=['hover'],
    xlabel='Observation Offset', ylabel='Value', title='Vital Aperiodic TS', subcoordinate_y=False, autorange='x',
    xlim=(xmin, xmax), width=720, height=480)

# Todo: Funktion bauen die das plottet, damit hier nicht so eine mega list comprehension rumliegt

overlay_subcoordinates = hv.Overlay(
    [hv.Curve((df.index, df[col]), label=str(col)) * hv.Scatter((df.index, df[col])).opts(size=5) for col in
     df.columns])
overlay_subcoordinates.opts(
    'Curve', tools=['hover'], xlabel='Observation Offset', ylabel='Value', title='Vital Aperiodic TS',
    subcoordinate_y=True, autorange='x', xlim=(xmin, xmax), width=720, height=480) 

vline_min = hv.VLine(xmin).opts(color='red', line_dash='dashed')
vline_max = hv.VLine(xmax).opts(color='red', line_dash='dashed')

layout_vital_aperiodic = (overlay_regular * vline_min * vline_max + overlay_subcoordinates * vline_min * vline_max).opts(
    shared_axes=False, normalize=False)
layout_vital_aperiodic

## Hospitals with data available

In [ ]:
query = query_schema + """
with t as
(
select distinct patientunitstayid
from vitalaperiodic
)
select 
  pt.hospitalid
  , count(distinct pt.patientunitstayid) as number_of_patients
  , count(distinct t.patientunitstayid) as number_of_patients_with_tbl
from patient pt
left join t
  on pt.patientunitstayid = t.patientunitstayid
group by pt.hospitalid
""".format(patientunitstayid)

df_available = pd.read_sql_query(query, engine)
df_available['data completion'] = df_available['number_of_patients_with_tbl'] / df_available[
    'number_of_patients'] * 100.0
df_available.sort_values('number_of_patients_with_tbl', ascending=False, inplace=True)
df_available.head(n=10)

In [ ]:
counts, edges = np.histogram(df_available["data completion"], bins=10)
histogram = hv.Histogram((counts, edges))
data_completion_histogram = histogram.opts(width=600, height=400, xlabel='Percent of patients with data (bins=10)', ylabel='Number of Records',
               title='Data Completion Histogram', tools=['hover'])
data_completion_histogram

Most hospitals in eICU have good data coverage for the vitalaperiodic table.

In [ ]:
import panel as pn
from panel.template import DarkTheme

#pn.extension(sizing_mode="stretch_width")
pn.extension()

panel_page = pn.template.BootstrapTemplate(title="Time Series Viewer", theme=DarkTheme, header_color="red")

#panel_from_holo = pn.pane.HoloViews(layout, sizing_mode="stretch_width")
#panel_from_holo = pn.pane.HoloViews(layout, sizing_mode="stretch_width")

panel_page.main.append(pn.Card(layout_vital_aperiodic, sizing_mode='scale_width'))
panel_page.main.append(pn.Card(data_completion_histogram, sizing_mode='scale_width'))

panel_page.show()

In [ ]:
from bokeh.resources import INLINE

panel_page.save('panel_exports/vital_aperiodic.html', resources=INLINE)